# 🚀 Task 5 — Predictive Model: Linear Regression
**DecodeLabs Data Analytics Internship | Batch 2026**  
**Intern:** Umm-e-Abiha

---

## 🎯 Objective
Design, train, and evaluate a supervised machine learning model to predict `TotalPrice` using customer and order features.

## 📋 Deliverables
- Feature engineering (label encoding, coupon flag)
- Train-test split (80/20)
- Linear Regression model training
- Model evaluation (R², RMSE)
- Feature importance analysis
- Sample prediction review
- Visualisation of model performance

## 🧠 Skills Applied
`scikit-learn` · `feature engineering` · `model evaluation` · `regression analysis`

---

## ⚙️ Environment Setup, Data Load & Clean

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Global Plot Style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor'  : '#1a1a2e',
    'axes.edgecolor'  : '#00d4ff',
    'axes.labelcolor' : '#ffffff',
    'xtick.color'     : '#cccccc',
    'ytick.color'     : '#cccccc',
    'text.color'      : '#ffffff',
    'grid.color'      : '#2a2a4a',
    'grid.linestyle'  : '--',
    'grid.alpha'      : 0.4,
})

CYAN   = '#00d4ff'
ORANGE = '#ff6b35'
GREEN  = '#39ff14'
PURPLE = '#bf5af2'
YELLOW = '#ffd60a'
PINK   = '#ff2d78'
COLORS = [CYAN, ORANGE, GREEN, PURPLE, YELLOW, PINK, '#00ff88']

print("✅ All libraries imported successfully!")

df_raw = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
df = df_raw.copy()
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')
df = df.drop_duplicates()
df['Date']      = pd.to_datetime(df['Date'])
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
for col in ['Product', 'OrderStatus', 'PaymentMethod', 'ReferralSource']:
    df[col] = df[col].str.strip().str.title()
print(f"Clean dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 5.1 Feature Engineering

In [ ]:
# ── Label Encoding for Categorical Features ───────────────────────────────────
df_model = df.copy()
le = LabelEncoder()

df_model['Product_enc']        = le.fit_transform(df_model['Product'])
df_model['PaymentMethod_enc']  = le.fit_transform(df_model['PaymentMethod'])
df_model['OrderStatus_enc']    = le.fit_transform(df_model['OrderStatus'])
df_model['ReferralSource_enc'] = le.fit_transform(df_model['ReferralSource'])
df_model['HasCoupon']          = (df_model['CouponCode'] != 'NO_COUPON').astype(int)

print("Feature Engineering Complete:")
print(f"   Product categories encoded   : {sorted(df_model['Product_enc'].unique())}")
print(f"   PaymentMethod encoded        : {sorted(df_model['PaymentMethod_enc'].unique())}")
print(f"   HasCoupon (0=No, 1=Yes)     : {df_model['HasCoupon'].value_counts().to_dict()}")


## 5.2 Define Features (X) and Target (y)

In [ ]:
# ── Feature & Target Selection ────────────────────────────────────────────────
features = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'HasCoupon', 'Month']
target   = 'TotalPrice'

X = df_model[features]
y = df_model[target]

print("Selected Features (X):")
for i, f in enumerate(features, 1):
    print(f"   {i}. {f}")
print(f"\nTarget Variable (y): {target}")
print(f"\nDataset shape: X={X.shape}, y={y.shape}")


## 5.3 Train-Test Split (80/20)

In [ ]:
# ── Train-Test Split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train-Test Split:")
print(f"   Training set  : {len(X_train):,} rows  ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test set      : {len(X_test):,} rows   ({len(X_test)/len(X)*100:.0f}%)")
print(f"   Random state  : 42 (reproducible split)")


## 5.4 Train the Linear Regression Model

In [ ]:
# ── Model Training ───────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

print("✅ Model training complete!")
print()
print("Model Equation (learned coefficients):")
print(f"   Intercept (bias) : {model.intercept_:.4f}")
print()
for feat, coef in zip(features, model.coef_):
    print(f"   {feat:<25} coefficient: {coef:>10.4f}")


## 5.5 Model Evaluation

In [ ]:
# ── Predictions & Metrics ────────────────────────────────────────────────────
y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 48)
print("   MODEL PERFORMANCE REPORT")
print("=" * 48)
print(f"   R² Score   : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print(f"   RMSE       : ${rmse:.2f}  (avg prediction error)")
print()
print("   Interpretation:")
print(f"   • The model explains {r2*100:.1f}% of the variance in TotalPrice")
print(f"   • On average, predictions are off by ${rmse:.2f}")
rating = "Excellent" if r2 > 0.85 else "Good" if r2 > 0.70 else "Fair"
print(f"   • Model rating: {rating}")
print("=" * 48)


## 5.6 Feature Importance (Absolute Coefficients)

In [ ]:
# ── Feature Importance ───────────────────────────────────────────────────────
feat_imp = pd.Series(np.abs(model.coef_), index=features).sort_values(ascending=False)

print("Feature Importance (|Coefficient|):")
print("─" * 50)
for feat, coef in feat_imp.items():
    bar = '█' * int(coef / feat_imp.max() * 25)
    print(f"  {feat:<25}  {coef:>8.2f}  {bar}")
print()
print(f"  Most influential feature: {feat_imp.idxmax()} ({feat_imp.max():.2f})")


## 5.7 Sample Predictions vs Actuals

In [ ]:
# ── Sample Prediction Review ─────────────────────────────────────────────────
print(f"{'Actual':>12}  {'Predicted':>12}  {'Error':>10}  {'Status'}")
print("─" * 52)
for actual, pred in list(zip(y_test.values, y_pred))[:12]:
    err    = actual - pred
    status = "✅ Accurate" if abs(err) < 300 else "⚠️  Review"
    print(f"  ${actual:>10.2f}  ${pred:>10.2f}  {err:>+10.2f}  {status}")


## 5.8 Model Performance Visualisation

In [ ]:
# ── Task 5 — 3-Panel Model Visualisation ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(f'Task 5 — Linear Regression Model  |  R² = {r2:.4f}  |  RMSE = ${rmse:.2f}',
             color=CYAN, fontsize=14, fontweight='bold')

# (a) Actual vs Predicted Scatter
axes[0].scatter(y_test, y_pred, color=CYAN, alpha=0.45, s=20, label='Predictions')
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], color=ORANGE, linestyle='--', lw=2, label='Perfect Fit')
axes[0].set_title(f'Actual vs Predicted\nR² = {r2:.4f}', color=CYAN, fontsize=12)
axes[0].set_xlabel('Actual Total Price ($)'); axes[0].set_ylabel('Predicted Total Price ($)')
axes[0].legend(fontsize=9)

# (b) Residuals Distribution
residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=40, color=PURPLE, edgecolor='#0d0d0d', alpha=0.85)
axes[1].axvline(0, color=ORANGE, linestyle='--', lw=2, label='Zero Error')
axes[1].set_title(f'Residuals Distribution\nRMSE = ${rmse:.2f}', color=CYAN, fontsize=12)
axes[1].set_xlabel('Residual (Actual − Predicted)'); axes[1].set_ylabel('Frequency')
axes[1].legend()

# (c) Feature Importance
feat_sorted = feat_imp.sort_values()
bar_colors  = [GREEN if v == feat_sorted.max() else CYAN for v in feat_sorted.values]
axes[2].barh(feat_sorted.index, feat_sorted.values, color=bar_colors, edgecolor='#0d0d0d')
axes[2].set_title('Feature Importance\n(|Coefficient|)', color=CYAN, fontsize=12)
axes[2].set_xlabel('Absolute Coefficient Value')

plt.tight_layout()
plt.savefig('task5_model.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("✅ Model charts saved → task5_model.png")


## 💡 Task 5 — Key Findings

### Model Summary
| Metric | Value | Interpretation |
|--------|-------|----------------|
| **R²** | ~0.89 | Model explains ~89% of variance in TotalPrice |
| **RMSE** | ~$288 | Average prediction error |
| **Rating** | Excellent | R² > 0.85 threshold |

### Feature Importance (Ranked)
1. **Quantity** — The dominant predictor; more units = higher order value
2. **Product_enc** — Product type significantly influences pricing
3. **HasCoupon** — Coupon usage has a measurable effect on order value
4. **UnitPrice** — Per-unit price drives total cost
5. **Month** — Minimal but present seasonal effect

### Business Recommendation
> **Quantity is the strongest revenue driver.** To increase average order value, implement bundle offers, volume discounts, and multi-unit promotions to incentivise customers to order more units per transaction.

### Residual Analysis
- Residuals follow an approximately normal distribution centred at zero
- Indicates model is unbiased and errors are random (good sign)
- Large residuals occur primarily for high-value outlier orders
